# 00 - Environment and unit tests
Records the exact Kaggle hardware/software identity and runs the unit tests. Kaggle numbers are **Kaggle** numbers, never Jetson numbers.

**Inputs:** none. **Outputs:** `/kaggle/working/artifacts/env/environment.json`, `status.json`, `unit_tests.txt`.

**Settings:** Internet ON, GPU on (T4/P100 is fine). Run all cells.

In [ ]:
# --- Setup: get the project code onto Kaggle (internet must be ON) ---
import os, sys, subprocess
REPO_URL = "https://github.com/Amith2277/Edge-Computing-EndSem.git"
REPO_DIR = "/kaggle/working/repo"
# If the repo is private or internet is off: attach the repo as a Kaggle dataset and set REPO_DIR to its path, e.g.
# REPO_DIR = "/kaggle/input/adaptivepark-code"
if not os.path.isdir(REPO_DIR):
    r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, universal_newlines=True)
    print(r.stdout)
    if r.returncode != 0:
        raise SystemExit("git clone failed. Turn Internet ON in notebook settings, or attach the repo as a dataset.")
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"])
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print("repo at", REPO_DIR)

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest", "pyyaml", "pillow"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, universal_newlines=True)
print(r.stdout[-500:])
# Do NOT upgrade torch/ultralytics here. Training notebooks pin ultralytics==8.3.40 explicitly.

In [ ]:
from pathlib import Path
from src.common.config import config_hash, load_yaml
from src.common.env_record import save_environment
from src.common.status import new_run_id, write_status

ART = Path("/kaggle/working/artifacts/env")
RUN_ID = new_run_id("env")
cfg_h = config_hash(load_yaml("configs/dataset.yaml"))
write_status(ART / "status.json", "00_environment", RUN_ID, "RUNNING", cfg_h)
env = save_environment(ART / "environment.json")
import json; print(json.dumps(env, indent=2))

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/unit"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, universal_newlines=True)
print(r.stdout[-3000:])
(ART / "unit_tests.txt").write_text(r.stdout)
state = "COMPLETE" if r.returncode == 0 else "FAILED"
write_status(ART / "status.json", "00_environment", RUN_ID, state, cfg_h, "unit tests " + ("passed" if r.returncode == 0 else "FAILED"))
print("STATE:", state)

## Next
Click **Save Version -> Save & Run All** if you want the outputs kept, then run notebook 01. Paste the printed environment and `STATE` back to the team/assistant.